In [ ]:
# CELL 1 — Runtime environment (read-only)
from pathlib import Path
import json, os, platform, shlex, shutil, subprocess, sys
import psutil
REPO = Path('/content/RFforNetwork_stage73b')
WINDOWS = Path('/content/datasets/CICIoMT24/WINDOWS_CPLUS')
LOCAL_RUNS = Path('/content/stage73b_runs')
DRIVE_ROOT = Path('/content/drive/MyDrive/COMPACT-MI-RFE')
print({'python':sys.version,'platform':platform.platform(),'cpu_logical':os.cpu_count(),'cpu_physical':psutil.cpu_count(logical=False),'ram_total':psutil.virtual_memory().total,'ram_available':psutil.virtual_memory().available,'disk_free':shutil.disk_usage('/content').free})
print('No training started. Do not use Run all.')

In [ ]:
# CELL 2 — Google Drive mount (manual opt-in)
MOUNT_DRIVE = False
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
else:
    print('Drive mount disabled.')

In [ ]:
# CELL 3 — Source identity; inspect an existing NEW checkout, never pull/reset/overwrite
CHECK_SOURCE_IDENTITY = False
if CHECK_SOURCE_IDENTITY:
    if not (REPO/'.git').is_dir(): raise FileNotFoundError(REPO)
    head = subprocess.run(['git','-C',str(REPO),'rev-parse','HEAD'],text=True,capture_output=True,check=True).stdout.strip()
    dirty = subprocess.run(['git','-C',str(REPO),'status','--porcelain'],text=True,capture_output=True,check=True).stdout
    if dirty.strip(): raise RuntimeError('Checkout is dirty; official run blocked.')
    sys.path.insert(0, str(REPO)); sys.path.insert(0, str(REPO/'huyen_code'))
    from colab.svm_official_readiness import normalized_source_sha256, source_contract
    print({'git_commit':head,'clean':True,'training_sha256_normalized':normalized_source_sha256(REPO/'huyen_code'/'ciciomt24_training.py'),'sliding_sha256_normalized':normalized_source_sha256(REPO/'huyen_code'/'ciciomt24_sliding_window.py'),'source_contract':source_contract(REPO)})
else:
    print('Source identity check disabled.')

In [ ]:
# CELL 4 — Dataset readiness; validate existing files only, never recreate windows
CHECK_DATASET_READINESS = False
if CHECK_DATASET_READINESS:
    sys.path.insert(0, str(REPO)); sys.path.insert(0, str(REPO/'huyen_code'))
    from colab.dataset_validator import validate_dataset
    DATASET_AUDIT = validate_dataset(WINDOWS, REPO, all_headers=True)
    print(json.dumps(DATASET_AUDIT, indent=2))
    print('Test feature rows read: 0; model.fit calls: 0')
else:
    print('Dataset readiness check disabled.')

In [ ]:
# CELL 5 — Unit tests in a separate Python process; no real dataset fit
RUN_STAGE73B_TESTS = False
UNIT_TEST_EVIDENCE = DRIVE_ROOT/'environment'/'stage73b_unit_tests.json'
if RUN_STAGE73B_TESTS:
    env = os.environ.copy(); env['PYTHONPATH'] = f'{REPO}:{REPO / "huyen_code"}'
    modules = ['colab.test_stage73b_authorization','colab.test_stage73b_supervisor','colab.test_stage73b_scientific_integrity','colab.test_svm_official_readiness','colab.test_stage6_colab','huyen_code.test_ciciomt24_svm_probe','huyen_code.test_ciciomt24_svm_scaling_probe']
    completed = subprocess.run([sys.executable,'-m','unittest','-v',*modules],cwd=REPO,env=env,text=True,capture_output=True)
    print(completed.stdout); print(completed.stderr)
    if completed.returncode != 0: raise RuntimeError('Tests failed; authorization remains blocked.')
    sys.path.insert(0,str(REPO)); from colab.dataset_validator import sha256
    evidence = {'passed':True,'modules':modules,'returncode':0,'training_sha256':sha256(REPO/'huyen_code'/'ciciomt24_training.py'),'full_training_authorized':False}
    UNIT_TEST_EVIDENCE.parent.mkdir(parents=True,exist_ok=True)
    if UNIT_TEST_EVIDENCE.exists(): raise FileExistsError(UNIT_TEST_EVIDENCE)
    UNIT_TEST_EVIDENCE.write_text(json.dumps(evidence,indent=2),encoding='utf-8')
    print('Evidence:',UNIT_TEST_EVIDENCE)
else:
    print('Stage 7.3B tests disabled.')

In [ ]:
# CELL 6 — Choose exactly one experiment/config/run
EXPERIMENT = 'baseline_svm'  # baseline_svm or mi_rfe_svm
CONFIG = 'FW_512_FS_16'
RUN_NAME = 'baseline_svm_FW512_FS16_official01'
APPROVAL = DRIVE_ROOT/'approvals'/f'{RUN_NAME}.json'
TECHNICAL_EVIDENCE = DRIVE_ROOT/'technical_evidence'/'REPLACE_WITH_APPROVED_FILE.json'
print({'experiment':EXPERIMENT,'config':CONFIG,'run_name':RUN_NAME,'default_gate':'BLOCKED'})

In [ ]:
# CELL 7 — Read-only official readiness; identity/template only, no fit
RUN_READ_ONLY_READINESS = False
if RUN_READ_ONLY_READINESS:
    sys.path.insert(0,str(REPO)); sys.path.insert(0,str(REPO/'huyen_code'))
    from colab.experiment_launcher import RunSpec
    from colab.svm_run_authorization import collect_identity, invalid_approval_template
    identity, evidence = collect_identity(REPO,WINDOWS,RunSpec(EXPERIMENT,CONFIG,RUN_NAME))
    print(json.dumps({'identity':identity,'evidence':evidence,'status':'BLOCKED_NO_EFFECTIVE_APPROVAL'},indent=2))
    blocked_template = DRIVE_ROOT/'approvals'/f'{RUN_NAME}.template.BLOCKED.json'
    print('Optional invalid template path:',blocked_template)
else:
    print('Readiness disabled; full training remains blocked.')

In [ ]:
# CELL 8 — Resource evidence review only; probe cannot authorize full training
REVIEW_RESOURCE_EVIDENCE = False
if REVIEW_RESOURCE_EVIDENCE:
    probe = json.loads(TECHNICAL_EVIDENCE.read_text(encoding='utf-8'))
    print(json.dumps({k:probe.get(k) for k in ['status','experiment','config','sizes','max_tree_rss_bytes','elapsed_seconds','resource_gate','full_training_authorized']},indent=2))
    if (probe.get('experiment'),probe.get('config')) != (EXPERIMENT,CONFIG):
        print('NON-EQUIVALENT EVIDENCE: explicit approved replacement policy is required.')
    print('No elapsed/RSS budget is inferred or approved by this cell.')
else:
    print('Resource evidence review disabled.')

In [ ]:
# CELL 9 — Approval review; default BLOCKED and no valid approval is generated
REVIEW_EFFECTIVE_APPROVAL = False
AUTHORIZATION_CONFIRMATION = ''
if REVIEW_EFFECTIVE_APPROVAL:
    sys.path.insert(0,str(REPO)); sys.path.insert(0,str(REPO/'huyen_code'))
    from colab.experiment_launcher import RunSpec
    from colab.svm_run_authorization import authorize_run
    unit = json.loads(UNIT_TEST_EVIDENCE.read_text(encoding='utf-8'))
    AUTH_CONTEXT = authorize_run(REPO,WINDOWS,LOCAL_RUNS,DRIVE_ROOT,RunSpec(EXPERIMENT,CONFIG,RUN_NAME),APPROVAL,TECHNICAL_EVIDENCE,AUTHORIZATION_CONFIRMATION,unit_test_evidence=unit)
    print(json.dumps(AUTH_CONTEXT.public_record(),indent=2))
else:
    print('BLOCKED: no effective run-specific approval reviewed.')

In [ ]:
# CELL 10 — Controlled execution entrypoint. DO NOT RUN during Stage 7.3B implementation.
START_OFFICIAL_SVM = False
I_ACCEPT_SINGLE_FINAL_TEST_EVALUATION = False
if not START_OFFICIAL_SVM or not I_ACCEPT_SINGLE_FINAL_TEST_EVALUATION:
    raise RuntimeError('Stage 7.3B execution remains BLOCKED; no training started.')
if not REVIEW_EFFECTIVE_APPROVAL or not AUTHORIZATION_CONFIRMATION.startswith('AUTHORIZE STAGE7.3B '):
    raise PermissionError('Read-only approval review and exact phrase are required.')
command = [sys.executable,'-m','colab.svm_official_supervisor','--repo',str(REPO),'--windows',str(WINDOWS),'--local-runs',str(LOCAL_RUNS),'--drive-root',str(DRIVE_ROOT),'--experiment',EXPERIMENT,'--config',CONFIG,'--run-name',RUN_NAME,'--approval',str(APPROVAL),'--technical-evidence',str(TECHNICAL_EVIDENCE),'--unit-test-evidence',str(UNIT_TEST_EVIDENCE),'--authorization-confirmation',AUTHORIZATION_CONFIRMATION]
env = os.environ.copy(); env['PYTHONPATH'] = f'{REPO}:{REPO / "huyen_code"}'
subprocess.run(command,cwd=REPO,env=env,check=True)

In [ ]:
# CELL 11 — Verify published metadata/artifact manifest; never evaluate Test again
VERIFY_COMPLETED_ARTIFACT = False
if VERIFY_COMPLETED_ARTIFACT:
    result_dir = DRIVE_ROOT/'experiment_results'/RUN_NAME
    result = json.loads((result_dir/'result.json').read_text(encoding='utf-8'))
    manifest = json.loads((result_dir/'artifact_manifest.json').read_text(encoding='utf-8'))
    if result.get('test_evaluation_calls') != 1: raise RuntimeError('Invalid Test evaluation count')
    required = {'result.json','model.joblib','final_scaler.joblib','authorization.json','hyperparameters.json','feature_schema.json'}
    if not required <= set(manifest.get('files',{})): raise RuntimeError('Artifact manifest incomplete')
    print({'status':result.get('status'),'files':len(manifest['files']),'test_evaluation_calls':1,'no_test_re_evaluation':True})
else:
    print('Artifact verification disabled.')

In [ ]:
# CELL 12 — Result summary from JSON only; no model load and no Test prediction
READ_RESULT_SUMMARY = False
if READ_RESULT_SUMMARY:
    result = json.loads((DRIVE_ROOT/'experiment_results'/RUN_NAME/'result.json').read_text(encoding='utf-8'))
    keys = ['status','experiment','config','rows_by_split','label_counts_by_split','metrics','elapsed_seconds','peak_rss_bytes_sampled','support_vectors','fit_warnings','label_origin','capture_session_independence','test_evaluation_calls']
    print(json.dumps({k:result.get(k) for k in keys},indent=2))
else:
    print('Summary disabled. Required return files: supervisor report/events, environment JSON, result.json, artifact_manifest.json, preflight_audit.json, authorization.json, hyperparameters.json, feature_schema.json, events.jsonl.')